# Figshare Document Pipeline

1. **DOI lookup** — batch-search Figshare for articles matching DOIs in `dois.txt`
2. **Article details** — fetch full metadata for each discovered article
3. **Local copy** — copy documents already present in the local backup
4. **Download** — download remaining documents directly from Figshare

In [1]:
import json
import tempfile
import time
from pathlib import Path

import requests
import shutil
import subprocess
from tqdm.auto import tqdm

BASE_URL = "https://api.figshare.com/v2"
DOIS_FILE = Path("dois.txt")
METADATA_DIR = Path("figshare_metadata")
DETAILS_DIR = Path("figshare_details")
MISSING_DOIS_FILE = METADATA_DIR / "missing_dois.txt"
BATCH_SIZE = 10

DEST_DIR = Path("pdfs")

DOCUMENT_EXTENSIONS = (".pdf", ".docx", ".doc")
MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds
TIMEOUT = 60  # seconds

METADATA_DIR.mkdir(parents=True, exist_ok=True)
DETAILS_DIR.mkdir(parents=True, exist_ok=True)
DEST_DIR.mkdir(exist_ok=True)

## 1. Load DOIs

In [2]:
dois = [
    line.strip()
    for line in DOIS_FILE.read_text().splitlines()
    if line.strip() and not line.startswith("#")
]
print(f"Loaded {len(dois):,} DOIs from {DOIS_FILE}")

Loaded 13,235 DOIs from dois.txt


## 2. Batch search Figshare for metadata

In [3]:
def _search_batch(doi_batch, page_size=100):
    """
    Search Figshare for articles whose related materials reference any DOI
    in *doi_batch*.  Returns the raw list of article search results.
    """
    query = " OR ".join(f":resource_doi: {d}" for d in doi_batch)
    data = {
        "search_for": query,
        "page_size": page_size,
        "order": "published_date",
        "order_direction": "desc",
    }
    resp = requests.post(f"{BASE_URL}/articles/search", json=data, timeout=TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def get_metadata_path(doi):
    return METADATA_DIR / f"{doi}.json"


def retrieve_metadata(dois):
    n_batches = (len(dois) + BATCH_SIZE - 1) // BATCH_SIZE
    not_found = []
    for i in tqdm(range(0, len(dois), BATCH_SIZE), total=n_batches, desc="Metadata"):
        batch = dois[i : i + BATCH_SIZE]

        try:
            articles = _search_batch(batch)
        except requests.RequestException as exc:
            tqdm.write(f"  ERROR — {exc}")
            time.sleep(2)
            continue

        per_doi = {doi: [] for doi in batch}
        for art in articles:
            doi = art["resource_doi"]
            if doi in per_doi:
                per_doi[doi].append(art)

        for doi, arts in per_doi.items():
            if arts:
                file_path = get_metadata_path(doi)
                file_path.parent.mkdir(parents=True, exist_ok=True)
                with open(file_path, "w") as f:
                    json.dump(arts, f, indent=2)
            else:
                not_found.append(doi)

        time.sleep(1)

    return not_found

In [4]:
missing_dois = (
    MISSING_DOIS_FILE.read_text().splitlines() if MISSING_DOIS_FILE.exists() else []
)
unseen_dois = [
    doi
    for doi in dois
    if doi not in missing_dois and not get_metadata_path(doi).exists()
]
not_found = retrieve_metadata(unseen_dois)
MISSING_DOIS_FILE.write_text("\n".join(sorted(missing_dois + not_found)))
print(
    f"Searched {len(unseen_dois):,} new DOIs, {len(unseen_dois) - len(not_found):,} found"
)
print(f"Metadata files: {len(list(METADATA_DIR.glob('**/*.json'))):,}")

Metadata: 0it [00:00, ?it/s]

Searched 0 new DOIs, 0 found
Metadata files: 11,260


## 3. Retrieve article details

In [5]:
def collect_article_ids(type_filter="journal contribution"):
    """Return a sorted list of unique article IDs from all metadata files.

    If *type_filter* is set, only include articles whose
    ``defined_type_name`` matches.
    """
    ids = set()
    for path in METADATA_DIR.glob("**/*.json"):
        with open(path) as f:
            for art in json.load(f):
                if type_filter and art.get("defined_type_name") != type_filter:
                    continue
                ids.add(art["id"])
    return sorted(ids)


def fetch_article_detail(article_id):
    resp = requests.get(f"{BASE_URL}/articles/{article_id}", timeout=TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def retrieve_details(article_ids):
    for aid in tqdm(article_ids, desc="Details"):
        dest = DETAILS_DIR / f"{aid}.json"
        if dest.exists():
            continue

        for attempt in range(MAX_RETRIES):
            try:
                detail = fetch_article_detail(aid)
                with open(dest, "w") as f:
                    json.dump(detail, f, indent=2)
                break
            except requests.RequestException as exc:
                tqdm.write(f"  {aid} attempt {attempt + 1} — {exc}")
                time.sleep(2 ** (attempt + 1))

        time.sleep(0.5)

In [6]:
article_ids = collect_article_ids()
unseen_ids = [aid for aid in article_ids if not (DETAILS_DIR / f"{aid}.json").exists()]
print(f"{len(article_ids):,} unique articles, {len(unseen_ids):,} still need details")

retrieve_details(unseen_ids)
print(f"Detail files: {len(list(DETAILS_DIR.glob('*.json'))):,}")

11,967 unique articles, 0 still need details


Details: 0it [00:00, ?it/s]

Detail files: 11,967


## 4. Download documents from Figshare

In [7]:
missing = []

for detail_file in sorted(DETAILS_DIR.glob("*.json")):
    with open(detail_file) as f:
        article = json.load(f)

    for file_entry in article.get("files", []):
        filename = file_entry["name"]
        if not filename.lower().endswith(DOCUMENT_EXTENSIONS):
            continue
        if (DEST_DIR / filename).exists():
            continue
        missing.append(
            {
                "filename": filename,
                "download_url": file_entry["download_url"],
                "article_id": detail_file.stem,
            }
        )

print(f"Documents already in {DEST_DIR}/: {len(list(DEST_DIR.iterdir())):,}")
print(f"Documents to download:           {len(missing):,}")

Documents already in pdfs/: 12,093
Documents to download:           0


In [8]:
downloaded = 0
failed = []

session = requests.Session()

for entry in tqdm(missing, desc="Downloading"):
    dest = DEST_DIR / entry["filename"]
    if dest.exists():
        downloaded += 1
        continue

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(entry["download_url"], timeout=TIMEOUT)
            resp.raise_for_status()
            with tempfile.NamedTemporaryFile(
                dir=DEST_DIR, delete=False, suffix=".tmp"
            ) as tmp:
                tmp.write(resp.content)
                tmp_path = Path(tmp.name)
            tmp_path.rename(dest)
            downloaded += 1
            break
        except Exception as exc:
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY * attempt)
            else:
                failed.append({**entry, "error": str(exc)})

print(f"\nDownloaded: {downloaded}")
print(f"Failed:     {len(failed)}")

Downloading: 0it [00:00, ?it/s]


Downloaded: 0
Failed:     0


In [9]:
if failed:
    print("Failed downloads:")
    for entry in failed:
        print(
            f"  {entry['filename']} (article {entry['article_id']}): {entry['error']}"
        )

## 5. Convert documents to PDF

In [10]:
libreoffice_available = shutil.which("libreoffice") is not None

converted = 0

if libreoffice_available:
    doc_files = sorted(
        p for p in DEST_DIR.iterdir() if p.suffix.lower() in (".doc", ".docx")
    )
    print(f"Documents to convert: {len(doc_files)}")

    conv_failed = []

    for src in tqdm(doc_files, desc="Converting"):
        pdf_dest = src.with_suffix(".pdf")
        if pdf_dest.exists():
            converted += 1
            continue

        try:
            with tempfile.TemporaryDirectory(dir=DEST_DIR) as tmp_dir:
                result = subprocess.run(
                    [
                        "libreoffice",
                        "--headless",
                        "--convert-to",
                        "pdf",
                        "--outdir",
                        tmp_dir,
                        str(src),
                    ],
                    capture_output=True,
                    text=True,
                    timeout=120,
                )
                tmp_pdf = Path(tmp_dir) / (src.stem + ".pdf")
                if result.returncode == 0 and tmp_pdf.exists():
                    tmp_pdf.rename(pdf_dest)
                    src.unlink()
                    converted += 1
                else:
                    conv_failed.append((src.name, result.stderr.strip()))
        except Exception as exc:
            conv_failed.append((src.name, str(exc)))

    print(f"Converted: {converted}")
    print(f"Failed:    {len(conv_failed)}")
    if conv_failed:
        for name, err in conv_failed:
            print(f"  {name}: {err}")

else:
    print("libreoffice not found in PATH; install it to convert .doc/.docx files")

Documents to convert: 129


Converting:   0%|          | 0/129 [00:00<?, ?it/s]

Converted: 129
Failed:    0


## 6. Summary

In [ ]:
print(f"DOIs loaded:    {len(dois):,}")
print(f"Metadata files: {len(list(METADATA_DIR.glob('**/*.json'))):,}")
print(f"Detail files:   {len(list(DETAILS_DIR.glob('*.json'))):,}")
print(f"Documents:      {len(list(DEST_DIR.iterdir())):,}")
print(f"Converted:      {converted}")

DOIs loaded:    13,235
Metadata files: 11,260
Detail files:   11,967
Documents:      12,093
Converted:      129
